### Data Ingestion


In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

C:\Users\avinash\AppData\Local\Temp\ipykernel_2160\891642371.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [15]:
docs=PyPDFLoader("C:\\Users\\avinash\\Desktop\\RAG\\data\\ilovepdf_merged (1).pdf")
docs=docs.load()



### Extract text

In [16]:
text="\n".join([doc.page_content for doc in docs])
print(len(text))


160766


### CREATE CHUNKS

In [17]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(docs)
print(f"split {len(docs)} documents into {len(chunks)} chunks")


split 57 documents into 212 chunks


### Embedding and VectorDatabase

In [4]:
from langchain_chroma import Chroma
print(Chroma)
print(hasattr(Chroma, 'from_documents'))

<class 'langchain_chroma.vectorstores.Chroma'>
True


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(
documents=chunks,
embedding=model,
persist_directory="./chroma_db"
) 


### Vectors

In [8]:
text=chunks[0].page_content
vector=model.embed_query(text)
print(len(vector))
print(vector)

384
[-0.036318667232990265, 0.04209234565496445, 0.0005191745585761964, -0.05446191504597664, -0.025263624265789986, -0.09619582444429398, -0.03584698587656021, 0.030096814036369324, 0.006160431541502476, -0.052785951644182205, -0.043593280017375946, -0.08591102808713913, 0.017772825434803963, -0.04530797526240349, -0.03875337168574333, -0.040253542363643646, -0.04631958156824112, -0.017269983887672424, -0.04709272086620331, -0.07885321974754333, 0.023470118641853333, -0.01334386970847845, -0.06504546105861664, 0.0074500408954918385, -0.0885881558060646, 0.0572756864130497, 0.0378088653087616, -0.029026981443166733, -0.06122523173689842, 0.005983762908726931, 0.027761735022068024, 0.012629435397684574, 0.040405675768852234, -0.04522433131933212, 0.0051806909032166, 0.16501370072364807, -0.0008990654023364186, -0.016974231228232384, 0.026825595647096634, 0.0014959096442908049, 0.05532639101147652, -0.11465172469615936, 0.036322783678770065, -0.02980007976293564, 0.005700960755348206, 0.

In [7]:
query="What is the drawback in existing systems?"
docs=vectorstore.similarity_search(query,k=5)
# for i ,doc in enumerate(docs,1):
#     print(f"Document {i}:")
#     print(doc.page_content)
#     print("\n")

In [8]:
retriver=vectorstore.as_retriever(search_kwargs={"k":25})
docs=retriver.invoke(query)


### Create a Prompt

In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the question using only the following context.
Context:{context},
Question:{question},
Answer:
""")

### API KEY

In [ ]:
import os
api_key=os.environ["OPENAI_API_KEY"]="YOUR_API_KEY_HERE"


### OPENAI LLM

In [11]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-5.6-luna")

### Generating the Answer using RAG PipeLine

In [19]:



while True:
    question=input("\nCHAT WITH ME:")
    if question.strip().lower() == "exit":
        print("Chat ended:")
        break
    docs=retriver.invoke(question)

    context="\n\n".join(doc.page_content for doc in docs)
    messages=prompt.invoke({
    "question":question,

    "context":context,
    })
    response=llm.invoke(messages)
print("\nAnswer")

print(response.content)

Chat ended:

Answer
The disadvantages of the existing system are:

- High susceptibility to human error and data manipulation.
- Delays in resource approval and water-quality reporting.
- Lack of real-time monitoring of water quality and usage patterns.
- Inefficient water allocation due to the absence of intelligent prediction models.
- Low transparency and accountability in water-management processes.
- Dependence on manual applications, sampling, and laboratory analysis.
- Centralized data storage with limited verification mechanisms.
- Decision-making based mainly on historical assumptions rather than real-time insights.
